<a href="https://colab.research.google.com/github/marvinjc/proyecto_final_CD/blob/Aaron/cd_final_proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto final: Aguas superficiales CONAGUA 2020
Limpieza, EDA, preparación (Pipeline) y K-means geográfico

Flujo OSEMN: Obtain → Scrub → Explore → Model → Interpret.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,FunctionTransformer

sns.set_theme(style="whitegrid")
RUTA_SITIOS = "/content/drive/MyDrive/CienciaDeDatos/ProyectoFinal/DataSets/Datos_de_calidad_del_agua_de_sitios_de_monitoreo_de_aguas_superficiales_2020.csv"
RUTA_ESCALAS = "/content/drive/MyDrive/CienciaDeDatos/ProyectoFinal/DataSets/Escalas_superficial.csv"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Carga e inspección inicial

OSEMN Obtain: se carga el CSV CONAGUA 2020 y se revisa forma, tipos y muestra.

In [ ]:
df_crudo = pd.read_csv(RUTA_SITIOS, encoding="latin-1")

In [ ]:
df_crudo.shape

(4141, 55)

In [ ]:
df_crudo.columns

Index(['CLAVE', 'SITIO', 'ORGANISMO_DE_CUENCA', 'ESTADO', 'MUNICIPIO',
       'CUENCA', 'CUERPO DE AGUA', 'TIPO', 'SUBTIPO', 'LONGITUD', 'LATITUD',
       'PERIODO', 'DBO_mg/L', 'CALIDAD_DBO', 'DQO_mg/L', 'CALIDAD_DQO',
       'SST_mg/L', 'CALIDAD_SST', 'COLI_FEC_NMP_100mL', 'CALIDAD_COLI_FEC',
       'E_COLI_NMP_100mL', 'CALIDAD_E_COLI', 'ENTEROC_NMP_100mL',
       'CALIDAD_ENTEROC', 'OD_PORC', 'CALIDAD_OD_PORC', 'OD_PORC_SUP',
       'CALIDAD_OD_PORC_SUP', 'OD_PORC_MED', 'CALIDAD_OD_PORC_MED',
       'OD_PORC_FON', 'CALIDAD_OD_PORC_FON', 'TOX_D_48_UT', 'CALIDAD_TOX_D_48',
       'TOX_V_15_UT', 'CALIDAD_TOX_V_15', 'TOX_D_48_SUP_UT',
       'CALIDAD TOX_D_48_SUP', 'TOX_D_48_FON_UT', 'CALIDAD_TOX_D_48_FON',
       'TOX_FIS_SUP_15_UT', 'CALIDAD_TOX_FIS_SUP_15', 'TOX_FIS_FON_15_UT',
       'CALIDAD_TOX_FIS_FON_15', 'SEMAFORO', 'CONTAMINANTES', 'CUMPLE_CON_DBO',
       'CUMPLE_CON_DQO', 'CUMPLE_CON_SST', 'CUMPLE_CON_CF',
       'CUMPLE_CON_E_COLI', 'CUMPLE_CON_ENTEROC', 'CUMPLE_CON_OD',
  

In [ ]:
df_crudo.dtypes

,0
CLAVE,object
SITIO,object
ORGANISMO_DE_CUENCA,object
ESTADO,object
MUNICIPIO,object
CUENCA,object
CUERPO DE AGUA,object
TIPO,object
SUBTIPO,object
LONGITUD,float64


In [ ]:
df_crudo.head()

,CLAVE,SITIO,ORGANISMO_DE_CUENCA,ESTADO,MUNICIPIO,CUENCA,CUERPO DE AGUA,TIPO,SUBTIPO,LONGITUD,...,CONTAMINANTES,CUMPLE_CON_DBO,CUMPLE_CON_DQO,CUMPLE_CON_SST,CUMPLE_CON_CF,CUMPLE_CON_E_COLI,CUMPLE_CON_ENTEROC,CUMPLE_CON_OD,CUMPLE_CON_TOX,GRUPO
0,DLAGU8,PRESA EL SAUCILLO 100M AGUAS ARRIBA DE LA CORTINA,LERMA SANTIAGO PACIFICO,AGUASCALIENTES,RINCON DE ROMOS,RIO SAN PEDRO,PRESA EL SAUCILLO,LENTICO,PRESA,-102.33911,...,"DQO,CF,",SI,NO,SI,NO,SI,ND,SI,SI,LENTICO
1,DLBAJ100,"LOS CABOS SEG 22, 2 ISA10B",PENINSULA DE BAJA CALIFORNIA,BAJA CALIFORNIA SUR,LOS CABOS,SAN JOSE DEL CABO,OCEANO PACIFICO,COSTERO,OCEANO-MAR,-109.84290,...,NaN,ND,ND,SI,ND,ND,SI,SI,SI,COSTERO
2,DLBAJ101,"LOS CABOS SEG 22, 1 ISA10B",PENINSULA DE BAJA CALIFORNIA,BAJA CALIFORNIA SUR,LOS CABOS,SAN LUCAS,OCEANO PACIFICO,COSTERO,OCEANO-MAR,-109.86442,...,NaN,ND,ND,SI,ND,ND,SI,SI,SI,COSTERO
3,DLBAJ102,LOS CABOS 3,PENINSULA DE BAJA CALIFORNIA,BAJA CALIFORNIA SUR,LOS CABOS,SAN LUCAS,BAHIA SAN LUCAS,COSTERO,BAHIA,-109.88604,...,NaN,ND,ND,SI,ND,ND,SI,SI,SI,COSTERO
4,DLBAJ103,LOS CABOS 1,PENINSULA DE BAJA CALIFORNIA,BAJA CALIFORNIA SUR,LOS CABOS,SAN LUCAS,BAHIA SAN LUCAS,COSTERO,BAHIA,-109.89657,...,NaN,ND,ND,SI,ND,ND,SI,SI,SI,COSTERO


In [ ]:
df_crudo.sample(5, random_state=42)

,CLAVE,SITIO,ORGANISMO_DE_CUENCA,ESTADO,MUNICIPIO,CUENCA,CUERPO DE AGUA,TIPO,SUBTIPO,LONGITUD,...,CONTAMINANTES,CUMPLE_CON_DBO,CUMPLE_CON_DQO,CUMPLE_CON_SST,CUMPLE_CON_CF,CUMPLE_CON_E_COLI,CUMPLE_CON_ENTEROC,CUMPLE_CON_OD,CUMPLE_CON_TOX,GRUPO
2083,OCGCE3239,FELIPE CARRILLO PUERTO,GOLFO CENTRO,VERACRUZ DE IGNACIO DE LA LLAVE,MARTINEZ DE LA TORRE,RIO NAUTLA,RIO BOBOS,LOTICO (HUMEDAL),RIO,-96.95467,...,"CF,E_COLI,",SI,SI,SI,NO,NO,ND,SI,SI,LOTICO
1965,OCFSU3124,"PUERTO MADERO SEG 73, 3 ISA8",FRONTERA SUR,CHIAPAS,TAPACHULA,PUERTO MADERO,OCEANO PACIFICO,COSTERO,OCEANO-MAR,-92.41338,...,NaN,ND,ND,SI,ND,ND,SI,SI,SI,COSTERO
1582,DLTAB5561,MANATI 10,FRONTERA SUR,TABASCO,MACUSPANA,Chilapa,NaN,LOTICO,NaN,-92.30760,...,"CF,",SI,SI,SI,NO,SI,ND,ND,SI,LOTICO
296,DLDUR676M1,PRESA SANTIAGO BAYACORA,PACIFICO NORTE,DURANGO,DURANGO,RIO SANTIAGO BAYACORA,PRESA SANTIAGO BAYACORA,LENTICO (HUMEDAL),PRESA,-104.67751,...,NaN,SI,SI,SI,SI,SI,ND,SI,SI,LENTICO
149,DLCHI294,PLANTA NORTE,RIO BRAVO,CHIHUAHUA,JUAREZ,RIO BRAVO 1,CANAL,LOTICO,CANAL,-106.37040,...,NaN,SI,SI,SI,SI,SI,ND,SI,SI,LOTICO


In [ ]:
print(df_crudo.shape)
df_crudo.info(verbose=True, show_counts=True)

(4141, 55)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4141 entries, 0 to 4140
Data columns (total 55 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CLAVE                   3493 non-null   object 
 1   SITIO                   3493 non-null   object 
 2   ORGANISMO_DE_CUENCA     3493 non-null   object 
 3   ESTADO                  3493 non-null   object 
 4   MUNICIPIO               3493 non-null   object 
 5   CUENCA                  3492 non-null   object 
 6   CUERPO DE AGUA          3479 non-null   object 
 7   TIPO                    3493 non-null   object 
 8   SUBTIPO                 3479 non-null   object 
 9   LONGITUD                3493 non-null   float64
 10  LATITUD                 3493 non-null   float64
 11  PERIODO                 3493 non-null   float64
 12  DBO_mg/L                2581 non-null   object 
 13  CALIDAD_DBO             2581 non-null   object 
 14  DQO_mg/L                2581 

In [ ]:
escalas = pd.read_csv(RUTA_ESCALAS, encoding="latin-1")
escalas.head(12)

,CALIDAD DEL AGUA PARA TOXICIDAD,CRITERIO,DESCRIPCION
0,No t¢xico,TOX menor a 1,Agua no contaminada.Toxicidad no detectable.
1,Toxicidad baja,TOX mayor o igual a 1 y menor o igual a 1.33,Toxicidad baja
2,Toxicidad moderada,TOX mayor de 1.33 y menor a 5,Toxicidad moderada
3,Toxicidad alta,TOX mayor o igual a 5,Toxicidad alta
4,CALIDAD DEL AGUA PARA SST,CRITERIO,DESCRIPCION
5,Excelente,SST menor o igual a 25,"Clase de excepci¢n, muy buena calidad."
6,Buena calidad,SST mayor de 25 y menor o igual a 75,Aguas superficiales con bajo contenido de soli...
7,Aceptable,SST mayor de 75 y menor o igual a 150,Aguas superficiales con indicio de contaminaci...
8,Contaminada,SST mayor de 150 y menor o igual a 400,Aguas superficiales de mala calidad con descar...
9,Fuertemente contaminada,SST mayor de 400,Aguas superficiales con fuerte impacto de desc...


## 2. Limpieza de datos
OSEMN Scrub: `a_numero`, LD/2 y nulos por GRUPO. No se pisa el CSV.

Rúbrica 2.5: Busqueda de nulos nulos, valores mal escritos (`<10`, `ND`) y sustitución (LD/2).

- `ND` / vacio -> NaN (no se midio; no se inventa un valor).
- `<10` -> 5: convencion (límite de detección)LD/2 para datos censurados. **No es la medicion exacta**; CONAGUA solo reporta que era menor al limite. Se usa para poder hacer describe, boxplot y correlacion.
- `>100` -> 100: se usa el limite; se pierde la distincion "mayor que".
- LATITUD / LONGITUD o SEMAFORO faltante: se elimina, **no se imputa**.

In [ ]:
nulos_antes = df_crudo.isna().sum()
nulos_antes = nulos_antes[nulos_antes > 0].sort_values(ascending=False)
pd.DataFrame({
    "nulos": nulos_antes,
    "porcentaje": (nulos_antes / len(df_crudo) * 100).round(1),
})

,nulos,porcentaje
TOX_FIS_FON_15_UT,4141,100.0
CALIDAD_TOX_FIS_FON_15,4141,100.0
TOX_D_48_FON_UT,4141,100.0
CALIDAD_TOX_D_48_FON,4141,100.0
OD_PORC_MED,3654,88.2
CALIDAD_OD_PORC_MED,3654,88.2
CALIDAD TOX_D_48_SUP,3379,81.6
TOX_D_48_SUP_UT,3379,81.6
ENTEROC_NMP_100mL,3237,78.2
CALIDAD_ENTEROC,3237,78.2


In [ ]:
Y = "SEMAFORO"
COLS_GEO = ["LONGITUD", "LATITUD"]
COLORES_SEMAFORO = {"Verde": "#2e7d32", "Amarillo": "#f9a825", "Rojo": "#c62828"}

COLS_LAB = [
    "DBO_mg/L", "DQO_mg/L", "SST_mg/L",
    "COLI_FEC_NMP_100mL", "E_COLI_NMP_100mL", "ENTEROC_NMP_100mL",
    "OD_PORC", "OD_PORC_SUP", "OD_PORC_MED", "OD_PORC_FON",
    "TOX_D_48_UT", "TOX_V_15_UT", "TOX_D_48_SUP_UT", "TOX_FIS_SUP_15_UT",
]

X_NUMERICAS = [
    "DBO_mg/L", "DQO_mg/L", "SST_mg/L",
    "COLI_FEC_NMP_100mL", "E_COLI_NMP_100mL", "ENTEROC_NMP_100mL",
    "OD_PORC", "OD_PORC_SUP",
]

def a_numero(valor):
    """'54.08' -> 54.08 | '<10' -> 5 (LD/2, convencion) | '>100' -> 100 | ND/vacio -> NaN"""
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip()
    if texto == "" or texto.upper() == "ND":
        return np.nan
    texto = texto.replace(",", "")
    if texto.startswith("<"):
        try:
            return float(texto[1:]) / 2.0
        except ValueError:
            return np.nan
    if texto.startswith(">"):
        try:
            return float(texto[1:])
        except ValueError:
            return np.nan
    try:
        return float(texto)
    except ValueError:
        return np.nan

In [ ]:
df = df_crudo.dropna(subset=["CLAVE"]).copy()
df["CLAVE"] = df["CLAVE"].astype(str).str.strip()
df = df[df["CLAVE"] != ""].copy()

for col in COLS_LAB + COLS_GEO:
    if col in df.columns:
        df[col] = df[col].map(a_numero)

df[Y] = df[Y].astype(str).str.strip().str.title()
df.loc[df[Y].isin(["Nan", "Nd", "None", ""]), Y] = np.nan

if "GRUPO" in df.columns:
    df["GRUPO"] = df["GRUPO"].astype(str).str.strip().str.upper()

n0 = len(df)
df = df.dropna(subset=COLS_GEO).copy()
print("Eliminadas por lat/lon faltante:", n0 - len(df))

n1 = len(df)
df = df.dropna(subset=[Y]).copy()
print("Eliminadas por SEMAFORO faltante:", n1 - len(df))
print("Sitios listos:", len(df))
df[COLS_LAB[:5] + COLS_GEO + [Y]].head()

Eliminadas por lat/lon faltante: 0
Eliminadas por SEMAFORO faltante: 0
Sitios listos: 3493


,DBO_mg/L,DQO_mg/L,SST_mg/L,COLI_FEC_NMP_100mL,E_COLI_NMP_100mL,LONGITUD,LATITUD,SEMAFORO
0,6.0,54.08,13.7500,1162.0,98.0,-102.33911,22.24730,Rojo
1,NaN,NaN,5.0000,NaN,NaN,-109.84290,22.90473,Verde
2,NaN,NaN,5.0000,NaN,NaN,-109.86442,22.89880,Verde
3,NaN,NaN,13.9667,NaN,NaN,-109.88604,22.89609,Verde
4,NaN,NaN,5.0000,NaN,NaN,-109.89657,22.87694,Verde


In [ ]:
nulos_despues = df.isna().sum()
nulos_despues = nulos_despues[nulos_despues > 0].sort_values(ascending=False)
pd.DataFrame({
    "nulos": nulos_despues,
    "porcentaje": (nulos_despues / len(df) * 100).round(1),
})

,nulos,porcentaje
TOX_FIS_FON_15_UT,3493,100.0
CALIDAD_TOX_FIS_FON_15,3493,100.0
CALIDAD_TOX_D_48_FON,3493,100.0
TOX_D_48_FON_UT,3493,100.0
OD_PORC_MED,3006,86.1
CALIDAD_OD_PORC_MED,3006,86.1
TOX_D_48_SUP_UT,2731,78.2
CALIDAD TOX_D_48_SUP,2731,78.2
CALIDAD_ENTEROC,2589,74.1
ENTEROC_NMP_100mL,2589,74.1


### Nulos por GRUPO
Comprobar si un parametro no se midio en COSTERO / LOTICO / LENTICO **antes** de afirmarlo.

In [ ]:
nulos_grupo = (
    df.groupby("GRUPO")[X_NUMERICAS]
    .agg(lambda serie: serie.isna().mean() * 100)
    .round(1)
)
nulos_grupo

,DBO_mg/L,DQO_mg/L,SST_mg/L,COLI_FEC_NMP_100mL,E_COLI_NMP_100mL,ENTEROC_NMP_100mL,OD_PORC,OD_PORC_SUP
GRUPO,,,,,,,,
COSTERO,91.3,91.3,0.1,91.3,91.3,9.4,94.8,9.0
LENTICO,1.0,1.0,0.0,0.8,0.8,99.2,100.0,2.2
LOTICO,0.2,0.2,0.2,0.2,0.2,99.8,1.5,99.8
